# **Code Architecture Documentation**

This is the structure of the main functioning part of the robot.

This version uses a Redis-based middleware for communication.

So, to understand TAGI at a deep technical level, the asynchronous process model must be checked and also how the Redis Key-Value store acts as the central nervous system. 

In this architecture, no two files talk to each other directly; they all talk to a shared memory space.

Inside elmo-v2's folder, the files seen on this document are located on a folder called ``/src``.

> #### Drivers

In elmo-v2, drivers are independent Linux processes. Their technical role is to manage the state machine of a specific peripheral and map physical registers to Redis keys.

Basically, they are used for low-level communication.

- ``driver_battery.py`` ::: Interacts with the power management IC via I2C to monitor voltage and current. It calculates the remaining capacity and updates the robot:status:battery key.

- ``driver_camera.py`` ::: It handles sensor initialization, gain control, and frame-buffer allocation to ensure high-speed image acquisition for vision tasks. The robot uses a Raspberry Pi 3 Camera Module.

- ``driver_led_matrix.py`` ::: A dedicated controller for the 13x13 RGB chest display. Converts color arrays into serialized SPI/bit-bang commands.

- ``driver_pan_tilt.py`` ::: The motion control for the head’s pan/tilt assembly. For further information about this library, always refer to ``elmo-v2/src/herkulex.py``.

- ``driver_screen.py`` ::: Controls the LCD graphical interface.

- ``driver_touch.py`` ::: Interfaces with the MPR121 capacitive touch controller. It performs signal filtering on five distinct electrodes to differentiate between momentary taps and sustained petting gestures.

- ``touch_calibrator.py`` ::: This is an utility to calibrate the MPR121 sensor. It measures the baseline capacitance of the robot's environment and calculates the optimal "release" and "touch" thresholds to ensure the sensors remain accurate regardless of humidity or surface interference.

> #### Behaviours

Behaviours are high-level scripts. They represent the personality of the robot.

- ``behaviour_blush.py`` ::: It triggers when driver_touch.py reports contact and it overrides the current LED matrix state to display a heart beat graph.

- ``behaviour_look_around.py`` ::: It sequences a radom pattern of head movement to maintain a "lifelike" presence and prevent mechanical seizing during inactivity.

- ``behaviour_safety_monitor.py`` ::: A high-priority supervisor process. It monitors internal diagnostics (thermals and battery) and can issue "Emergency Stop" commands to the drivers if hardware thresholds are exceeded.

> #### Tools and Utilities

Tools provide the necessary utilities for system configuration, external networking, and specialized communication protocols.

- ``httpserver.py`` ::: An asynchronous REST API gateway that serves as the primary bridge for external command-and-control, translating incoming HTTP requests into internal Redis messages.

- ``loadconfig.py`` ::: It parses ``config.yaml`` to populate the Redis master-dictionary with persistent parameters such as servo offsets, network credentials and gain constants.

- ``middleware.py`` ::: The middleware is the whole orchestrator of the robot; it initializes the Redis connection, manages the global state dictionary, and provides the primary loop that synchronizes data flow between asynchronous driver processes and synchronous behavior logic. For more information, check the [Middleware Documentation](middleware.ipynb).

- ``mjpeg_server_2.py`` ::: An optimized version of the original MJPEG tool. It implements improved threading or a different web framework (such as ``Tornado`` or ``FastAPI``) to handle multiple concurrent video stream requests from different companion apps simultaneously.

- ``robot_api.py`` ::: It wraps the Redis commands into a structured Python class, providing a clean programmatic interface for the ``httpserver.py`` to get/set robot states without writing raw Redis queries.